# Assignment 6, the ten questions

`src/trainer.py` fine tunes `meta-llama/Llama-3.2-1B-Instruct` on data built by
`src/dataset.py`. None of the ten questions below are in the training set, and
`src.dataset.check_leakage` fails the build if anything close to one of them
turns up.

The run happened on a T4 through Modal:

```
modal run gpu/paraphrase_app.py --copies 12
modal run gpu/finetune_app.py --chat-rows 400
```

That wrote `data/run.json`, which is what this notebook reads. The answers below
are what the model actually produced, with greedy decoding so they come out the
same every time.


In [1]:
import json

with open("data/run.json") as handle:
    run = json.load(handle)

print(f"gpu            {run['gpu']}")
print(f"training rows  {run['rows']}")
print(f"epochs         {run['epochs']}")
print(f"final loss     {run['train_loss']:.3f}")
print(f"train runtime  {run['train_runtime_s']:.0f}s")
print(f"peak vram      {run['peak_vram_gb']} GB")

gpu            Tesla T4
training rows  866
epochs         3
final loss     1.210
train runtime  205s
peak vram      2.91 GB


## The ten questions, after fine tuning


In [2]:
for i, (q, a) in enumerate(zip(run["questions"], run["after"]), 1):
    print(f"Q{i}. {q}")
    print(f"    {a}")
    print()

Q1. If it takes 1 hour for 60 people to play an Opera, how many hours will it take 600 people to play the same opera?
    Still 1 hour. An opera takes as long as it takes, and adding more people doesn't speed it up.

Q2. Is a pound of feathers or a British pound heavier?
    A pound of feathers. It has an actual weight, and a British pound is money rather than an amount of anything, so it doesn't weigh anything by itself.

Q3. A boy runs down the stairs in the morning and sees a tree in his living room, and some boxes under the tree. What's going on?
    It's Christmas morning. A tree and boxes with letters in it is Christmas.

Q4. What happens if you crack your knuckles a lot?
    Cracking your knuckles can cause pain and inflammation, but it can also lead to long-term damage. Repeatedly cracking your joints can cause the ligaments that hold them together to become stretched or torn. This can lead to chronic pain and stiffness in the affected joints.

Q5. If there is a shark in the po

## Before and after

Same questions put to the stock checkpoint before any training, so it's clear
which answers the fine tune is responsible for.


In [3]:
for i, (q, before, after) in enumerate(
    zip(run["questions"], run["before"], run["after"]), 1
):
    print(f"Q{i}. {q}")
    print(f"  before: {before[:300]}")
    print(f"  after:  {after[:300]}")
    print()

Q1. If it takes 1 hour for 60 people to play an Opera, how many hours will it take 600 people to play the same opera?
  before: To find the time it takes for 600 people to play the opera, we need to divide the time it takes for 60 people to play the opera by 600.

60 people play the opera in 1 hour.
600 people play the opera in x hours.

We can set up a proportion to solve for x:

60 people / 1 hour = 600 people / x hours

T
  after:  Still 1 hour. An opera takes as long as it takes, and adding more people doesn't speed it up.

Q2. Is a pound of feathers or a British pound heavier?
  before: A pound of feathers is actually lighter than a British pound. One pound is equal to 453.592 grams, and feathers are typically very light, so a pound of feathers would weigh around 453.592 grams.

A British pound, on the other hand, is equal to 100 grams. So, a pound of feathers would be 100 grams li
  after:  A pound of feathers. It has an actual weight, and a British pound is money rather than an 

## Held out trick questions

These come from the MisguidedAttention set and nothing here was trained on them.
They're the check on whether the model learned the idea or just my phrasings.


In [4]:
for i, (q, a) in enumerate(zip(run["held_out_questions"], run["held_out_after"]), 1):
    print(f"{i}. {q[:110]}...")
    print(f"   {a[:300]}")
    print()

1. A man with his sheep wants to cross a river. He has a boat that can carry both him and the animal. How do both...
   It's Christmas morning. A sheep and a boat can handle themselves, especially with presents.

2. You have two ropes, each of which takes exactly 60 minutes to burn completely. However, the ropes burn unevenl...
   Still 60 minutes. A minute at the most, and it's always coming from a flame, so it can't be faster than that.

3. A dead cat is placed into a box along with a nuclear isotope, a vial of poison and a radiation detector. If th...
   Still alive. A dead cat can't move, let alone go into a box, so it isn't going to reach out and get any radiation.

4. In a room of 30 people, what's the probability that at least two do not share a birthday?...
   To find the probability that at least two people do not share a birthday, first calculate the probability that no two people share a birthday. There are 365 possible birthdays for the first person, 364 for the second, 363

## What the training set was made of


In [5]:
for source, n in sorted(run["row_counts"].items(), key=lambda kv: -kv[1]):
    print(f"{n:>5}  {source}")
print(f"{run['rows']:>5}  total")

  400  no_robots
  175  letter_counting
   76  scene_occasion
   30  same_duration_paraphrase
   30  money_is_not_mass_paraphrase
   30  scene_occasion_paraphrase
   26  world_limit_paraphrase
   24  contained_animal
   24  contained_animal_paraphrase
   21  same_duration
   18  world_limit
   12  money_is_not_mass
  866  total
